### Задача 1 

    По тропинке вдоль кустов
    Шли 11 хвостов.
    Сосчитать я также смог,
    Что шагало 30 ног.
    Это вместе шли куда-то
    Петухи и поросята.
    А теперь вопрос таков:
    Сколько было петухов?
    И узнать я был бы рад,
    Сколько было поросят?

Неизвестным в задаче является количество петухов и поросят, запишем это в виде множеств целых чисел:

In [1]:
from z3 import * # подключаем все возможности

roosters, piglets = Ints('roosters piglets')

Теперь сформулируем ограничения. Двигаемся по порядку задачи. 
Первое что известно "По тропинке вдоль кустов шли 11 хвостов". Сейчас у нас определны множества петухов и поросят, надо связать атрибут "хвост" с сортами поросёнок и петух. Если преположить что среди них небыло мутантов, то 1 петуху или поросенку принадлежит строго 1 хвост :) Если хвостов было 11 то и животных 11, значит: 


In [2]:
tails_constraint = roosters + piglets == IntVal(11)

Также известно, что "шагало 30 ног". Знаем, что у поросят 4 ноги, а у петухов 2 и делаем вывод, что выражение описывающее ограничение должно выглядеть так:

In [3]:
legs_constraint = roosters * 2 + piglets * 4 == IntVal(30)

Количество животных не может быть отрицательным, поэтому накладываем дополнительные ограничения на количество.
Знаем, что шагали петухи и поросята, следовательно должен быть хотябы 1 петух и хотябы 1 поросёнок. Формализуем это ограничение:

In [4]:
quantity_limit = And(roosters>=1, piglets>=1)

Теперь можно обратиться к солверу за решением:

In [5]:
solve(tails_constraint, legs_constraint, quantity_limit)

[piglets = 4, roosters = 7]


### Задача 2

Есть три разработчика:

Алиса
Боб
Чарли

Нужно определить, кто сегодня выходит в офис.

Известно:

Хотя бы один должен прийти.
Алиса и Боб не могут прийти одновременно.
Если Чарли приходит, то Алиса тоже приходит.
Боб сегодня не придет.



Определим булевые переменные, которые обозначают кто сегодня пришёл в офис, а кто нет:

In [6]:
Alice, Bob, Charlie = Bools('Alice Bob Charlie')

Запишем ограничения:

In [7]:
solver = Solver()

solver.add(Or(Alice, Bob, Charlie))
solver.add(Not(And(Alice, Bob)))
solver.add(Implies(Charlie, Alice))
solver.add(Not(Bob))

In [8]:
solver.check()

sat

In [9]:
solver.model()

[Charlie = False, Alice = True, Bob = False]

### Задача 3

Три студента:

Анна
Борис
Вика

Заняли первое, второе и третье места.

Известно:

У каждого свое место.
Анна не первая.
Борис выше Вики.
Вика не последняя.

Нужно найти кто какое место занял. Определим Анну, бориса и Викторию в виде списка целочисленных переменных, значением переменной будет место участника:

In [10]:
solver = Solver()

Anna, Boris, Vika = Ints('Anna Boris Vika')
places = [Anna, Boris, Vika]

Поскольку мест всего 3 и они уникальны, то зададим соответсвующие ограничения - значение места с 1 по 3, и их уникальность:

In [11]:
solver.add([And(i >= 1, i <= 3) for i in places]) # Ограничиваем диапазон.
solver.add(Distinct(places)) # Указываем, что значения должны быть уникальны.

Учитываем, что Анна не первая:

In [12]:
solver.add(Anna != 1)

Учитываем, что Борис выше Вики:

In [13]:
solver.add(Boris < Vika)

Учитываем, что Вика не последняя:

In [14]:
solver.add(Vika != 3)

In [15]:
solver.check()

sat

In [16]:
solver.model()

[Anna = 3, Vika = 2, Boris = 1]

### Задача 4

Три человека: Алиса, Боб, Чарли

Один всегда говорит правду. Один всегда лжет. Один может говорить что угодно.
Они говорят:
- Алиса: «Боб — лжец.»
- Боб: «Чарли говорит правду.»
- Чарли: «Я не непредсказуемый.»

Определить:
кто правдивый;
кто лжец;
кто непредсказуемый.

Определим константы для описания роли человека - лжец, правдивый или непредсказуемый:

In [39]:
truth = IntVal(1)
liar = IntVal(2)
random = IntVal(3)

Определим переменные которые будут хранить целочисленно значение роли и обозначать соответствующего человека:

In [40]:
Alice, Bob, Charlie = Ints('Alice Bob Charlie')
persons = [Alice, Bob, Charlie]

Зададим первое ограничение - каждый из них может иметь только одну роль:

In [41]:
solver = Solver()

solver.add(Distinct(persons))

Теперь зададим диапазон допустимых значений:

In [42]:
solver.add([And(p >= 1, p <= 3) for p in persons])

Теперь запишем для каждого их утверждения возможные условия. Начнём с Алисы:

In [43]:
solver.add(Implies(Alice == truth, Bob == liar)) # Если Алиса говорит правду.
solver.add(Implies(Alice == liar, Not(Bob == liar))) # Если Алиса лжёт.

Теперь Боб:

In [44]:
solver.add(Implies(Bob == truth, Charlie == truth)) # Если Боб говорит правду.
solver.add(Implies(Bob == liar, Not(Charlie == truth))) # Если Боб лжёт.

Не забываем про Чарли:

In [48]:
solver.add(Implies(Charlie == truth, Not(Charlie == random))) # Если Чарли говорит правду.
solver.add(Implies(Charlie == liar, Charlie == rabdom)) # Если Чарли лжёт.

In [49]:
solver.check()

sat

In [50]:
solver.model()

[Alice = 2, Bob = 3, Charlie = 1]

### Задача 5

Есть четыре разработчика:

Анна
Борис
Виктор
Глеб

Они должны выполнить четыре задачи:

Backend
Frontend
DevOps
QA

Каждый выполняет ровно одну задачу.
Все задачи распределены между разными людьми.
Анна не занимается DevOps.
Борис занимается задачей раньше Виктора (Backend = 1, Frontend = 2, DevOps = 3, QA = 4).
Глеб не Backend.
Виктор не QA.

Для решения определим доступные роли как целочисленные значения:

In [24]:
backend = IntVal(1)
frontend = IntVal(2)
devops = IntVal(3)
qa = IntVal(4)

roles = [backend, frontend, devops, qa]

А также разработчиков как целочисленные переменные. Значение переменной и будет ролью:

In [25]:
Anna, Boris, Viktor, Gleb = Ints("Anna Boris Viktor Gleb")
developers = [Anna, Boris, Viktor, Gleb]

Опишем известные ограничения. 
Известно что каждый выполняет только одну задачу, значит значения в списке разработчиков будут уникальными:

In [26]:
solver = Solver()
solver.add(Distinct(developers))

Также нужно указать, что диапазон значений лежит строго в рамках спитска roles:

In [27]:
solver.add([Or([d == r for r in roles]) for d in developers])

Учтём что Анна не devops, Глеб не backend, а Виктор не qa:

In [28]:
solver.add(And(Anna != devops, Gleb != backend, Viktor != qa))

Также учтём, что Борис занимается задачей раньше Виктора:

In [29]:
solver.add(Boris < Viktor)

Отмечу, что кроме значения роли числовое значение значит еще и порядок исполнения. Но нам не нужно их выыводить в таком порядке поскольку задача лишь убедиться что имя разработчика и его роль соответсвуют всем условиям выше, поэтому проверяем возможность решения:

In [30]:
solver.check()

sat

In [31]:
solver.model()

[Anna = 4, Gleb = 2, Viktor = 3, Boris = 1]